# Responsible AI: explainability & disaggregated evaluation

Inspect global feature importance and accuracy **by facility** and **by
snapshot day**. **All data is synthetic.** See `docs/governance/responsible-ai.md`.

In [ ]:
from revenue_prediction.config.loader import load_settings
from revenue_prediction.data.synthetic import generate_synthetic_dataset
from revenue_prediction.training.splitting import blocked_temporal_split
from revenue_prediction.training.train import train_candidate

settings = load_settings('dev')
df = generate_synthetic_dataset(settings.data)
split = blocked_temporal_split(df, settings.split)
result = train_candidate('hist_gradient_boosting', split, settings.model, settings.features)
result.metrics

In [ ]:
from revenue_prediction.data.schema import FEATURE_COLUMNS
from revenue_prediction.evaluation.explainability import permutation_feature_importance

keep = [c for c in FEATURE_COLUMNS if c in split.test.columns]
X_test = result.feature_builder.transform(split.test[keep])
y_test = split.test['actual_month_end_net_revenue']
importance = permutation_feature_importance(result.estimator, X_test, y_test, n_repeats=5)
importance.head(15)

## SHAP explanations (optional)

SHAP gives consistent, additive attributions and mirrors the AutoML / Responsible AI dashboard explanations. Requires the `explain` extra: `uv sync --extra explain`.


In [ ]:
from revenue_prediction.evaluation.explainability import shap_summary

# Model-agnostic SHAP on the held-out test set (skip for naive baselines).
shap_summary(result.estimator, X_test, max_samples=150, top=15)

In [ ]:
print('Accuracy by facility:')
print(result.by_facility.to_string(index=False))
print('\nAccuracy by snapshot day (does it improve later in the month?):')
print(result.by_snapshot_day.to_string(index=False))

Review the Responsible AI checklist before any production promotion. This is
operational decision support on synthetic data — not clinical decision support,
financial advice, or an autonomous decision system.